# Notebook 02 — Transform, Segment, and Analyze Churn
## Contoso Banque — Churn Analysis Workshop

> **All data used in this notebook is entirely synthetic and fictional.**

---

## What This Notebook Does

This notebook reads the 5 raw Delta tables created by Notebook 01 and produces two curated analytical tables:

| Output | Description |
|---|---|
| `customer_360` | One row per customer — all features combined: demographics, balance metrics, product count, transaction activity, segments, and churn label |
| `churn_by_segment` | Aggregated churn KPIs grouped by segment (activity tier, balance band, product count tier, income band) |

## Prerequisites
- Notebook 01 must have run successfully.
- `ChurnAnalysisLH` must be attached as the Lakehouse.
- Tables: `customers`, `accounts`, `products`, `customer_products`, `transactions` must exist.

## Cell 1 — Imports and Setup

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, count, sum as spark_sum, avg, when, lit, round as spark_round,
    coalesce, max as spark_max, min as spark_min
)

spark = SparkSession.builder.getOrCreate()

# Reference date — must match Notebook 01
from datetime import date, timedelta
REFERENCE_DATE = date(2024, 3, 31)
DATE_90_DAYS_AGO = str(REFERENCE_DATE - timedelta(days=90))

print("✅ Spark session ready")
print(f"   Reference date: {REFERENCE_DATE}")
print(f"   90-day window start: {DATE_90_DAYS_AGO}")

## Cell 2 — Load Source Tables

In [ ]:
customers_df       = spark.read.table("customers")
accounts_df        = spark.read.table("accounts")
transactions_df    = spark.read.table("transactions")
customer_products_df = spark.read.table("customer_products")

print("✅ Source tables loaded:")
print(f"   customers:         {customers_df.count():>10,}")
print(f"   accounts:          {accounts_df.count():>10,}")
print(f"   transactions:      {transactions_df.count():>10,}")
print(f"   customer_products: {customer_products_df.count():>10,}")

## Cell 3 — Compute Account-Level Aggregates

For each customer, aggregate account information:
- Total and average balance across all accounts
- Dominant balance trend (most common trend across accounts)

In [ ]:
from pyspark.sql.functions import first

# Account-level aggregation per customer
account_agg = accounts_df.groupBy("customer_id").agg(
    count("account_id").alias("account_count"),
    spark_round(spark_sum("avg_balance_90d"), 2).alias("total_balance_90d"),
    spark_round(avg("avg_balance_90d"), 2).alias("avg_balance_90d"),
    spark_round(avg("current_balance"), 2).alias("avg_current_balance"),
    first("balance_trend").alias("balance_trend"),  # dominant trend
)

print(f"✅ Account aggregates: {account_agg.count():,} customers with account data")
account_agg.show(5)

## Cell 4 — Compute Product Counts per Customer

In [ ]:
product_agg = customer_products_df.groupBy("customer_id").agg(
    count("product_id").alias("total_product_count"),
    spark_sum("active_product_flag").alias("active_product_count"),
)

print(f"✅ Product aggregates: {product_agg.count():,} customers with product data")
product_agg.show(5)

## Cell 5 — Compute Transaction Metrics per Customer (Last 90 Days)

In [ ]:
from pyspark.sql.functions import to_date, abs as spark_abs

# Filter to last-90-day transactions
recent_txn = transactions_df.filter(
    col("transaction_date") >= DATE_90_DAYS_AGO
)

txn_agg = recent_txn.groupBy("customer_id").agg(
    count("transaction_id").alias("transaction_count_90d"),
    spark_round(
        spark_sum(when(col("amount") < 0, spark_abs(col("amount"))).otherwise(lit(0))),
        2
    ).alias("total_spend_90d"),
    spark_round(avg(spark_abs(col("amount"))), 2).alias("avg_txn_amount_90d"),
)

print(f"✅ Transaction aggregates: {txn_agg.count():,} customers with recent transactions")

## Cell 6 — Build Customer 360 Table

Join all aggregates to the customers base table, then apply segmentation logic.

In [ ]:
# Join all aggregates to customers
c360 = customers_df \
    .join(account_agg,  on="customer_id", how="left") \
    .join(product_agg,  on="customer_id", how="left") \
    .join(txn_agg,      on="customer_id", how="left")

# Fill nulls for customers with no accounts / products / transactions
c360 = c360 \
    .fillna({"account_count": 0, "total_balance_90d": 0.0, "avg_balance_90d": 0.0,
             "avg_current_balance": 0.0}) \
    .fillna({"total_product_count": 0, "active_product_count": 0}) \
    .fillna({"transaction_count_90d": 0, "total_spend_90d": 0.0, "avg_txn_amount_90d": 0.0}) \
    .fillna({"balance_trend": "Stable"})

print(f"Customer 360 before segmentation: {c360.count():,} rows")
c360.printSchema()

## Cell 7 — Apply Segmentation Logic

Add three segmentation dimensions based on clear, business-driven thresholds:

| Dimension | Logic |
|---|---|
| `activity_tier` | Based on transaction count in last 90 days |
| `balance_band` | Based on average balance in last 90 days |
| `product_count_tier` | Based on number of active products |

In [ ]:
# Activity tier segmentation
c360 = c360.withColumn(
    "activity_tier",
    when(col("transaction_count_90d") == 0, "Inactive")
    .when(col("transaction_count_90d") <= 3, "Low")
    .when(col("transaction_count_90d") <= 9, "Medium")
    .otherwise("High")
)

# Balance band segmentation (EUR)
c360 = c360.withColumn(
    "balance_band",
    when(col("avg_balance_90d") >= 25000, "High")
    .when(col("avg_balance_90d") >= 5000,  "Medium")
    .when(col("avg_balance_90d") >= 500,   "Low")
    .otherwise("Very Low")
)

# Product count tier segmentation
c360 = c360.withColumn(
    "product_count_tier",
    when(col("active_product_count") == 0, "No product")
    .when(col("active_product_count") == 1, "Single")
    .when(col("active_product_count") == 2, "Dual")
    .otherwise("Multi-product")
)

# Select final columns for customer_360
customer_360_df = c360.select(
    "customer_id",
    "age",
    "region",
    "tenure_months",
    "income_band",
    "risk_profile",
    "digital_active_flag",
    "churned_90d",
    "avg_balance_90d",
    "balance_trend",
    "active_product_count",
    "transaction_count_90d",
    "total_spend_90d",
    "activity_tier",
    "balance_band",
    "product_count_tier",
)

print(f"✅ Customer 360 final: {customer_360_df.count():,} rows")

# Segment distribution sanity check
print("\nActivity tier distribution:")
customer_360_df.groupBy("activity_tier").count().orderBy("count", ascending=False).show()

print("Balance band distribution:")
customer_360_df.groupBy("balance_band").count().orderBy("count", ascending=False).show()

## Cell 8 — Compute Churn KPIs by Segment

Build the `churn_by_segment` table — aggregated churn KPIs for each segmentation dimension.

In [ ]:
from pyspark.sql import Row

def compute_segment_kpis(df, dimension_col, dimension_name):
    """Compute churn KPIs for a given segmentation column."""
    return df.groupBy(dimension_col).agg(
        count("customer_id").alias("total_customers"),
        spark_sum("churned_90d").alias("churned_customers"),
        spark_round(100.0 * spark_sum("churned_90d") / count("customer_id"), 2).alias("churn_rate_pct"),
        spark_round(avg("avg_balance_90d"), 2).alias("avg_balance"),
        spark_round(avg("active_product_count"), 2).alias("avg_product_count"),
        spark_round(avg("transaction_count_90d"), 2).alias("avg_transaction_count"),
    ).withColumn("segment_dimension", lit(dimension_name)) \
     .withColumnRenamed(dimension_col, "segment_value")

# Compute for each dimension
seg_activity = compute_segment_kpis(customer_360_df, "activity_tier",     "activity_tier")
seg_balance  = compute_segment_kpis(customer_360_df, "balance_band",      "balance_band")
seg_products = compute_segment_kpis(customer_360_df, "product_count_tier", "product_count_tier")
seg_income   = compute_segment_kpis(customer_360_df, "income_band",        "income_band")

# Union all segments
churn_by_segment_df = seg_activity.unionByName(seg_balance) \
    .unionByName(seg_products) \
    .unionByName(seg_income) \
    .select(
        "segment_dimension",
        "segment_value",
        "total_customers",
        "churned_customers",
        "churn_rate_pct",
        "avg_balance",
        "avg_product_count",
        "avg_transaction_count",
    )

print(f"✅ Churn by segment: {churn_by_segment_df.count():,} segment rows")
churn_by_segment_df.orderBy("churn_rate_pct", ascending=False).show(20)

## Cell 9 — Write Output Delta Tables

In [ ]:
print("Writing curated Delta tables...")

customer_360_df.write.format("delta").mode("overwrite").saveAsTable("customer_360")
print("  ✅ Delta table: customer_360")

churn_by_segment_df.write.format("delta").mode("overwrite").saveAsTable("churn_by_segment")
print("  ✅ Delta table: churn_by_segment")

print("\n✅ All curated tables written.")

## Cell 10 — Validation and Business Summary

Final sanity checks and a summary of key churn insights.

In [ ]:
print("=" * 60)
print("VALIDATION — Final Table Counts")
print("=" * 60)

for tname in ["customers", "accounts", "products", "customer_products",
               "transactions", "customer_360", "churn_by_segment"]:
    n = spark.sql(f"SELECT COUNT(*) AS n FROM {tname}").collect()[0][0]
    print(f"  {tname:<30} {n:>10,}")

print()
print("=" * 60)
print("BUSINESS SUMMARY — Churn Insights")
print("=" * 60)

# Overall KPI
spark.sql("""
    SELECT
        COUNT(*) AS total_customers,
        SUM(churned_90d) AS churned,
        ROUND(100.0 * SUM(churned_90d) / COUNT(*), 2) AS overall_churn_rate_pct
    FROM customer_360
""").show()

# Top churn segments
print("Top segments by churn rate (min 50 customers):")
spark.sql("""
    SELECT segment_dimension, segment_value, total_customers, churned_customers, churn_rate_pct
    FROM churn_by_segment
    WHERE total_customers >= 50
    ORDER BY churn_rate_pct DESC
    LIMIT 10
""").show(truncate=False)

print("✅ Notebook 02 complete.")
print("   Next steps:")
print("   → Run SQL queries from sql/02_descriptive_churn_queries.sql")
print("   → Build Power BI report (see powerbi/report_design.md)")
print("   → Configure Data Agent (see docs/data_agent_setup.md)")